# Auditing components.yaml

This notebook runs the AuditSystem against the main components.yaml file to evaluate solution quality.

In [ ]:
from pathlib import Path

import ibis as ib
from ibis import _

from iacs.architect import Architect

ib.options.interactive = True  # auto-display results (like pandas)

## Load components.yaml

In [ ]:
COMPONENTS_DIR = "../builtins"

a = Architect.from_manifest(COMPONENTS_DIR)

## Evaluate highest-priority tasks to work on

In [ ]:
entity_ids = a.get("entity_id")
entity_ids

In [ ]:
parents = a.get("parent")
parents

In [ ]:
reqs = a.get("requirement")
reqs

In [ ]:
(
    reqs
    .join(parents, reqs.entity_id == parents.parent_id, how="anti")
    .join(entity_ids.select(_.value, _.entity_key, _.path), _.entity_id == entity_ids.value )
    .order_by(ib.desc(_.priority))
)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Configure: entity_key of the ancestor requirement to visualize
ANCESTOR_KEY = "be_a_powerful_tool_for_solutions_architecture"

# Pull data to pandas for graph construction
entity_ids_pd = entity_ids.to_pandas()
parents_pd = parents.to_pandas()
reqs_pd = reqs.to_pandas()

# Lookups
id_to_key = entity_ids_pd.set_index("value")["entity_key"].to_dict()
req_ids = set(reqs_pd["entity_id"].unique())

# Find the ancestor hash
ancestor_rows = entity_ids_pd[entity_ids_pd["entity_key"] == ANCESTOR_KEY]
if ancestor_rows.empty:
    raise ValueError(f"No entity found with entity_key '{ANCESTOR_KEY}'")
ancestor_id = ancestor_rows.iloc[0]["value"]

# Build a full parent->child graph and collect all descendants
G_full = nx.DiGraph()
for _, row in parents_pd.iterrows():
    G_full.add_edge(row["parent_id"], row["entity_id"])

descendants = nx.descendants(G_full, ancestor_id) | {ancestor_id}

# Keep only nodes that are requirements
req_nodes = descendants & req_ids | {ancestor_id}

# Build the requirement subtree
G = nx.DiGraph()
for _, row in parents_pd.iterrows():
    if row["parent_id"] in req_nodes and row["entity_id"] in req_nodes:
        G.add_edge(row["parent_id"], row["entity_id"])
if ancestor_id not in G:
    G.add_node(ancestor_id)

labels = {n: id_to_key.get(n, n[:8]) for n in G.nodes()}


def hierarchy_pos(G, root, width=1.0, vert_gap=0.2, vert_loc=0.0, xcenter=0.5):
    def _pos(G, node, width, vert_gap, vert_loc, xcenter, pos, parent):
        pos[node] = (xcenter, vert_loc)
        children = [n for n in G.successors(node) if n != parent]
        if children:
            dx = width / len(children)
            x = xcenter - width / 2 + dx / 2
            for child in children:
                _pos(G, child, dx, vert_gap, vert_loc - vert_gap, x, pos, node)
                x += dx
        return pos
    return _pos(G, root, width, vert_gap, vert_loc, xcenter, {}, None)


pos = hierarchy_pos(G, ancestor_id, width=3.0, vert_gap=0.25)

fig, ax = plt.subplots(figsize=(18, 12))
nx.draw(
    G, pos, labels=labels, ax=ax,
    node_size=1800, node_color="steelblue", font_size=6.5,
    font_color="white", arrows=True, arrowsize=12,
    edge_color="gray", width=1.2,
)
ax.set_title(f"Requirements tree: {ANCESTOR_KEY}", fontsize=13)
plt.tight_layout()
plt.show()

## Requirements tree visualization